In [0]:
%pip install transformers
%pip install torch
%pip install torchvision
%pip install tensorflow==2.17.0

In [0]:
%restart_python

In [0]:
%sql
CREATE or replace TABLE workspace.default.product_reviews (
  review_id STRING,
  customer_name STRING,
  rating INT,
  review_text STRING,
  review_date DATE
)
USING DELTA;


In [0]:
%sql
select * from workspace.default.product_reviews;

In [0]:
%sql
INSERT INTO workspace.default.product_reviews VALUES
-- Review 1
('r001', 'John D', 5, 
'This product exceeded my expectations.

The build quality is top-notch and feels very premium in hand. I was particularly impressed by the packaging — it was neat, secure, and had zero damage.

I have been using it for over a week now and there have been no issues whatsoever. Highly recommended!', 
'2024-07-01'),

-- Review 2
('r002', 'Sara M', 3, 
'The product is decent for the price.

However, I was disappointed by the packaging. The box was partially torn and one corner of the item had scratches.

Overall, it works fine, but I expected better presentation.', 
'2024-07-02'),

-- Review 3
('r003', 'Amit K', 1, 
'Very bad experience.

First of all, I received a completely different product than what I ordered. The support team took 4 days to respond, and their replies were copy-paste templates.

Ultimately, I had to return the item. Waste of time and energy.', 
'2024-07-03'),

-- Review 4
('r004', 'Priya G', 4, 
'Good product but with a few issues.

The quality is satisfactory and the features match the description. But delivery took longer than expected — almost 8 days.

Customer support was polite and kept me updated. Overall, a positive experience.', 
'2024-07-04'),

-- Review 5
('r005', 'Alex B', 2, 
'Not very happy with the purchase.

Although the design looks sleek online, the actual product looks cheaper. Also, the instruction manual was missing.

It might be okay for casual use, but I wouldn’t buy it again.', 
'2024-07-05');


In [0]:
%sql
SELECT * FROM workspace.default.product_reviews;

In [0]:
%restart_python

In [0]:
from transformers import pipeline

summarizer = pipeline("text-generation", model="t5-small")
text = "Databricks makes data teams more productive with a unified analytics platform."
summary = summarizer(text, min_length=20, max_length=50)
print(summary)


In [0]:
# Use the actual output instead of generate_signature_output
#output = summary
#signature = infer_signature(text, output)

In [0]:
import mlflow
from mlflow.models import infer_signature

# It is valuable to log a "signature" with the model, telling MLflow the input and output schema
output = summary
signature = infer_signature(text, output)
print(f"Signature:\n{signature}\n")

# Set experiment path (visible in sidebar under Machine Learning -> Experiments)
experiment_name = f"/Workspace/Users/santi1947@gmail.com/GenAI/poc/GenAIExperiment001"
mlflow.set_experiment(experiment_name)

model_artifact_path = "summarizer"  # Name of folder containing serialized model




In [0]:
with mlflow.start_run():
    # LOG PARAMS
    mlflow.log_params({
        "hf_model_name": "t5-small"
    })
    mlflow.transformers.log_model(
        transformers_model=summarizer,
        artifact_path=model_artifact_path,
        input_example=text,
        signature=signature
 )

In [0]:
# Grab most recent run (which logged the model) using the experiment name
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
runs = mlflow.search_runs([experiment_id])

# Get the latest run by start time (descending order)
last_run_id = runs.sort_values('start_time', ascending=False).iloc[0].run_id

# Construct the model URI from that run ID and artifact path
model_uri = f"runs:/{last_run_id}/{model_artifact_path}"
print(f"Model URI: {model_uri}")


In [0]:
# Load the model using transformers loader to avoid pyfunc wrapper incompatibility
loaded_summarizer = mlflow.transformers.load_model(model_uri=model_uri)

# Run prediction on text input
summary_output = loaded_summarizer(text)
print("Generated Summary:\n", summary_output)


In [0]:
from mlflow import MlflowClient

# Define model nam using Unity Catalog syntax
model_name = "workspace.default.summarizer_model"

# Point to Unity Catalog as the model registry backend
mlflow.set_registry_uri("databricks-uc")

# Register the model using the URI from a previous run
mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedModelInput, ServedModelInputWorkloadSize

w = WorkspaceClient()

# Update the endpoint to use model version 2
w.serving_endpoints.update_config(
    name="demo",
    served_models=[
        ServedModelInput(
            model_name="workspace.default.summarizer_model",
            model_version="2",
            workload_size=ServedModelInputWorkloadSize.SMALL,
            scale_to_zero_enabled=True
        )
    ]
)

print("Endpoint 'demo' updated to serve model version 2")
print("The endpoint will now redeploy with TensorFlow 2.17.0 (compatible dependencies)")

In [0]:
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

# Get endpoint details
endpoint = w.serving_endpoints.get("demo")

print(f"\n{'='*60}")
print(f"ENDPOINT: {endpoint.name}")
print(f"{'='*60}")
print(f"State: {endpoint.state.config_update}")
print(f"Ready: {endpoint.state.ready}")

# Show served models
if endpoint.config and endpoint.config.served_models:
    print(f"\nServed Models:")
    for model in endpoint.config.served_models:
        print(f"  - {model.model_name} v{model.model_version}")
        print(f"    Workload Size: {model.workload_size}")
        print(f"    Scale to Zero: {model.scale_to_zero_enabled}")

# Show pending config if updating
if endpoint.pending_config:
    print(f"\nPending Configuration:")
    for model in endpoint.pending_config.served_models:
        print(f"  - {model.model_name} v{model.model_version}")

print(f"\n{'='*60}")
print(f"Recent Build/Deployment Events:")
print(f"{'='*60}")

# Get recent events (without specifying served model name since it's updating)
try:
    events = w.serving_endpoints.logs("demo")
    event_list = list(events)[:10]
    if event_list:
        for i, event in enumerate(event_list):
            print(f"\n[{i+1}] {event.event_type}")
            print(f"    Time: {event.timestamp}")
            print(f"    Message: {event.message}")
    else:
        print("\nNo events available yet.")
except Exception as e:
    print(f"\nCouldn't retrieve detailed logs: {str(e)[:100]}")
    print("This is normal during deployment - check back in a few minutes.")

In [0]:

from mlflow.tracking import MlflowClient
client = MlflowClient()
client.set_registered_model_alias(alias='Production_model',
    name="summarizer_model",
    version=1
)


In [0]:
df=spark.read.table("workspace.default.product_reviews")
df.display()

In [0]:
loaded_model=mlflow.pyfunc.load_model(model_uri=model_uri)

In [0]:
loaded_model

In [0]:
df2=df.toPandas()
# Load model using transformers loader to avoid pyfunc wrapper incompatibility
transformers_model = mlflow.transformers.load_model(model_uri=model_uri)
summaries = df2['review_text'].apply(lambda text: transformers_model(text)[0]['generated_text'])

In [0]:
summaries

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

model_name='summarizer_model'
# Load model using transformers loader to avoid pyfunc wrapper incompatibility
transformers_model = mlflow.transformers.load_model(model_uri=f"models:/{model_name}@production_model")

@pandas_udf("string")
def prod_model_udf(texts: pd.Series) -> pd.Series:
    return texts.apply(lambda text: transformers_model(text)[0]['generated_text'])


In [0]:
# Load model inside UDF to avoid serialization - capture only the model URI string
from pyspark.sql.functions import pandas_udf
import pandas as pd

# Use the full Unity Catalog model name
model_uri_str = "models:/workspace.default.summarizer_model@production_model"

@pandas_udf("string")
def batch_model_udf(texts: pd.Series) -> pd.Series:
    import mlflow
    # Set Unity Catalog as registry and load model inside the UDF (once per executor)
    mlflow.set_registry_uri("databricks-uc")
    model = mlflow.transformers.load_model(model_uri=model_uri_str)
    return texts.apply(lambda text: model(text)[0]['generated_text'])

batch_df = df.withColumn('summary', batch_model_udf('review_text'))
display(batch_df)

In [0]:
batch_df.write.mode('append').saveAsTable("workspace.default.batch_summary")

In [0]:
%sql
CREATE OR REPLACE TABLE ai_query_inference AS (
  SELECT
    review_id,
    ai_query(
      "databricks-llama-4-maverick",
      CONCAT(
        "Based on the following document, provide a summary in less than 100 words. Document: ",
        review_text
      )
    ) AS generated_summary
  FROM workspace.default.batch_summary
  LIMIT 10
);


In [0]:
%sql
select * from ai_query_inference